#### Building a Chatbot

In this video, we'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:
* Conversational RAG: Enable a chatbot experience over an external source of data.
* Agents: Build a chatbot that can take actions

This section will cover the basics which will be helpful for those two more advanced topics.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3.6-27b", groq_api_key=groq_api_key)
model

d:\Anaconda3\envs\LangChain_Langgraph_Course\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x00000234456FBF50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000234456CCB90>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is Harshit and I am a Senior Machine Learning Engineer")])

AIMessage(content="\n<think>\nHere's a thinking process:\n\n1.  **Analyze User Input:**\n   - **Name:** Harshit\n   - **Role/Title:** Senior Machine Learning Engineer\n   - **Tone:** Professional, introductory\n\n2.  **Identify Key Elements to Acknowledge:**\n   - Greet the user by name\n   - Acknowledge their role/title\n   - Set a professional yet friendly tone\n   - Offer assistance relevant to their field\n\n3.  **Determine Response Goals:**\n   - Welcome Harshit\n   - Acknowledge his senior ML engineering role\n   - Express readiness to help with ML/AI topics, architecture, coding, research, career advice, etc.\n   - Keep it concise and open-ended\n\n4.  **Draft Response (Mental Refinement):**\n   Hi Harshit! Nice to meet you. It’s great to connect with a Senior ML Engineer. Whether you’re looking to discuss model architecture, MLOps, scaling pipelines, cutting-edge research, or even career/tech leadership topics, I’m here to help. What can I assist you with today?\n\n5.  **Check 

In [6]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, My name is Harshit and I am a Senior Machine Learning Engineer"),
        AIMessage(content="Hi Harshit! Nice to meet you. 👋 It's great to connect with a Senior ML Engineer. Whether you're looking to dive into model architecture, MLOps & scaling, experiment tracking, production deployment, cutting-edge research, or even tech leadership & strategy, I'm here to help.\n\nWhat are you working on right now, or how can I assist you today?"),
        HumanMessage(content="Hey. What is my name and what do I do?")
    ]
)

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hey. What is my name and what do I do?"\n   - This is a direct question asking me to recall information they provided in the previous message.\n\n2.  **Review Context:**\n   - In the first message, the user said: "Hi, My name is Harshit and I am a Senior Machine Learning Engineer"\n   - I acknowledged this in my response.\n\n3.  **Formulate Response:**\n   - I need to accurately recall and state:\n     - Name: Harshit\n     - Role: Senior Machine Learning Engineer\n   - Keep it concise and friendly.\n   - Example: "Your name is Harshit, and you\'re a Senior Machine Learning Engineer!"\n\n4.  **Self-Correction/Verification:**\n   - Does it match the context? Yes.\n   - Is it accurate? Yes.\n   - Tone appropriate? Yes, professional and friendly.\n   - No extra fluff needed.\n\n   Ready. Output matches response.✅\n</think>\n\nYour name is **Harshit**, and you\'re a **Senior Machine L

#### Message History

We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some database. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!!

In [11]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [12]:
config = {"configurable": {"session_id":"chat1"}}

In [14]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Harshit and I am a Senior Machine Learning Engineer")],
    config=config
)

In [16]:
with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config
)

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What is my name?"\n2.  **Check Context/History:** In the previous messages, the user explicitly stated: "Hi, My name is Harshit and I am a Senior Machine Learning Engineer" (twice, actually).\n3.  **Identify Key Information:** The name provided is "Harshit".\n4.  **Formulate Response:** Acknowledge the name clearly and directly. Keep it concise and friendly.\n   - Draft: "Your name is Harshit!"\n   - Add a polite/engaging follow-up if needed, but keep it direct.\n5.  **Verify:** Does it match the context? Yes. Is it accurate? Yes. Is it concise? Yes.\n   - Final: "Your name is Harshit! How can I help you today?" (or similar)\n   - I\'ll keep it simple and direct.✅\n</think>\n\nYour name is Harshit! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 219, 'prompt_tokens': 254, 'total_tokens': 473, 'completion_time': 0.415747226, 

In [19]:
config1 = {"configurable":{"session_id": "chat1"}}
response = with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config1
)
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user is asking "What is my name?" for the fourth time in a row.\n2.  **Check Context:** In previous turns, the user explicitly stated: "Hi, My name is Harshit and I am a Senior Machine Learning Engineer". I\'ve consistently acknowledged this and answered "Your name is Harshit!" each time.\n3.  **Identify Pattern/Intent:** The user is likely testing my memory/context retention, or just repeating the question to see if I maintain consistency. It\'s a straightforward factual question based on the provided context.\n4.  **Formulate Response:** Keep it concise, accurate, and consistent with previous responses. Acknowledge the name clearly. Optionally, add a brief prompt to move the conversation forward since they keep asking the same question.\n5.  **Draft Response:** Your name is Harshit! How can I help you today? (or similar)\n6.  **Self-Correction/Refinement:** The response is already optimal. No extra fluff neede

In [22]:
# In this case the model won't know anything because there is no history for session_id=chat2
config2 = {"configurable":{"session_id": "chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config2
)
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user is asking "What is my name?" for the second time.\n2.  **Identify Core Constraint:** I am an AI without persistent memory across sessions, and I don\'t have access to personal/user data unless explicitly provided in the current conversation.\n3.  **Check Context:** In the previous turn, I already explained that I don\'t know their name and offered to be told what to call them. The user repeated the exact same question.\n4.  **Determine Response Strategy:** \n   - Acknowledge the repeated question\n   - Reiterate clearly that I don\'t have access to personal information or memory\n   - Keep it concise and friendly\n   - Offer to help if they share their name or ask something else\n5.  **Draft Response (Mental):** I still don\'t know your name since I don\'t store personal information or remember past conversations. If you\'d like to tell me what to call you, I\'d be happy to! Otherwise, how can I help you to

#### Prompt Templates

Prompt templates help to turn raw user information into a  format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, Let's add in a system message with some custom instructions (but still taking messages as input). Next, we will add in more input besides just the messages.

In [35]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all the questions to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model

In [36]:
response = chain.invoke({"messages":[HumanMessage("Hi. My name is Harshit")]})

In [38]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [39]:
config = {"configurable": {"session_id": "chat3"}}
response = with_message_history.invoke(
    [HumanMessage(content="Hi. My name is Harshit")],
    config=config
)

response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi. My name is Harshit"\n   - This is a simple greeting and introduction.\n   - I already responded to this exact message in the previous turn, but the user repeated it. This could be a mistake, a test, or they just want to continue the conversation.\n\n2.  **Identify Key Elements:**\n   - Greeting: "Hi."\n   - Name: "Harshit"\n   - Context: Repeated message\n\n3.  **Determine Response Strategy:**\n   - Acknowledge the greeting and name again politely.\n   - Keep it friendly and open-ended.\n   - Ask how I can assist them.\n   - Avoid sounding repetitive or robotic.\n\n4.  **Draft Response (Mental):**\n   Hi Harshit! It\'s great to meet you. How can I assist you today? Whether you have questions, need help with something, or just want to chat, I\'m here for you!\n\n5.  **Refine Response:**\n   - Keep it concise and warm.\n   - Match the tone of the user.\n   - Ensure it\'s ready f

In [40]:
## Let's make this a bit more complex

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all the questions to the best of your ability in {language}"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model

In [41]:
response = chain.invoke({"messages":[HumanMessage("Hi. My name is Harshit")], "language": "Hindi"})
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi. My name is Harshit"\n   - Language: English\n   - Key information: Greeting + Name introduction\n\n2.  **Identify Constraints:**\n   - System prompt: "Answer all the questions to the best of your ability in Hindi"\n   - This means I must respond in Hindi, regardless of the input language.\n\n3.  **Determine Response Content:**\n   - Acknowledge the greeting\n   - Use the user\'s name (Harshit)\n   - Respond politely and warmly in Hindi\n   - Keep it concise\n\n4.  **Draft Response (Mental Translation to Hindi):**\n   - English: Hello Harshit! Nice to meet you. How can I assist you today?\n   - Hindi: नमस्ते हर्षित! आपसे मिलकर अच्छा लगा। आज मैं आपकी कैसे मदद कर सकता हूँ?\n\n5.  **Check Against Constraints:**\n   - Is it in Hindi? Yes.\n   - Does it address the user appropriately? Yes.\n   - Is it polite and helpful? Yes.\n\n6.  **Final Output Generation:** (matches the drafted Hindi response)\n 

In [43]:
with_message_history = RunnableWithMessageHistory(
    chain, 
    get_session_history,
    input_messages_key="messages"
)

d:\Anaconda3\envs\LangChain_Langgraph_Course\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [45]:
config = {"configurable": {"session_id": "chat4"}}
response = with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi. I am Harshit")], "language": "Hindi"},
    config=config
)
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi. I am Harshit"\n   - Language: English\n   - Intent: Greeting and self-introduction\n   - System Prompt: "Answer all the questions to the best of your ability in Hindi"\n\n2.  **Identify Key Constraints:**\n   - Must respond in Hindi\n   - Should acknowledge the greeting and the name\n   - Should be polite and helpful\n\n3.  **Formulate Response (Mental Translation & Refinement):**\n   - English: "Hello Harshit! Nice to meet you. How can I help you today?"\n   - Hindi: "नमस्ते हर्षित! आपसे मिलकर अच्छा लगा। आज मैं आपकी कैसे मदद कर सकता हूँ?"\n   - Check tone: Polite, friendly, matches the constraint (Hindi)\n   - Check accuracy: Correctly uses the name, standard greeting, offers help\n\n4.  **Final Output Generation:**\n   - Output matches the refined Hindi response.\n   - Ensure it strictly follows the "answer in Hindi" directive.\n   - Keep it concise.\n\n   "नमस्ते हर्षित! आपसे मिलकर अच्छा लगा

#### Managing the Conversation History

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages.

In [54]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer = trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I am bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="What is 2+2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hi! I am bob', additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwarg

In [55]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)
    | prompt
    | model
)

response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What ice cream do I like")],
        "language": "English"
    }   
)

response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What ice cream do I like"\n2.  **Scan Conversation History:** I need to look back at the conversation to find any mention of ice cream preferences.\n   - User: "hi! I am bob"\n   - Assistant: "hi!"\n   - User: "I like vanilla ice cream"\n   - Assistant: "nice"\n   - User: "What is 2+2"\n   - Assistant: "4"\n   - User: "thanks"\n   - Assistant: "no problem!"\n   - User: "having fun?"\n   - Assistant: "yes!"\n   - User: "What ice cream do I like"\n3.  **Identify Key Information:** In the second message, the user explicitly stated: "I like vanilla ice cream"\n4.  **Formulate Response:** Based on the conversation history, I should answer that they like vanilla ice cream. Keep it concise and friendly, matching the tone.\n5.  **Check Constraints:** The prompt says "Answer all the questions to the best of your ability in English you\'re a good assistant". No specific format required, just direct and accurate

In [56]:

response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What math problem did I ask")],
        "language": "English"
    }   
)

response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What math problem did I ask"\n2.  **Scan Conversation History:**\n   - User: "hi! I am bob"\n   - Model: "hi!"\n   - User: "I like vanilla ice cream"\n   - Model: "nice"\n   - User: "What is 2+2"\n   - Model: "4"\n   - User: "thanks"\n   - Model: "no problem!"\n   - User: "having fun?"\n   - Model: "yes!"\n   - User: "What math problem did I ask"\n3.  **Identify the Math Problem:** The only math problem asked was "What is 2+2"\n4.  **Formulate Response:** Keep it concise and direct, matching the conversational tone. "You asked what 2+2 is." or "You asked \'What is 2+2?\'"\n5.  **Check Constraints:** The prompt says "Answer all the questions to the best of your ability in English". No special constraints otherwise. Keep it simple.\n6.  **Draft Response:** You asked "What is 2+2".✅\n   Matches history, direct, accurate. Ready. \n   Output matches response.✅\n   Proceed. \n   *Self-Correction/Refinement 

In [57]:
## Lets wrap this in the message history

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
config = {"configurable": {"session_id": "chat5"}}

d:\Anaconda3\envs\LangChain_Langgraph_Course\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [60]:

response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="What ice cream do I like")],
        "language": "English"
    }  ,
    config = config 
)

response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "What ice cream do I like"\n   - This is a personal question about the user\'s preferences.\n   - I have no prior context or memory of the user\'s preferences.\n   - I\'m an AI without access to personal data unless explicitly provided in the conversation.\n\n2.  **Identify Key Constraints:**\n   - I don\'t have memory of past conversations (unless in the same session, but this is the first prompt).\n   - I can\'t know personal preferences without being told.\n   - I should respond honestly, politely, and offer to help if they tell me their preference.\n\n3.  **Formulate Response:**\n   - Acknowledge the question\n   - Explain that I don\'t have access to personal information or memory of their preferences\n   - Ask them to share what they like so I can remember it for this conversation or help them explore options\n   - Keep it friendly and helpful\n\n   Draft: "I don’t actually know what ice cream